In [1]:
from PyLTSpice import SimRunner, SpiceEditor, LTspice
from PyLTSpice import RawRead
import numpy as np
from scipy.integrate import simpson
import copy

In [2]:
runner = SimRunner(output_folder='./SimResults', simulator=LTspice)
netlist = SpiceEditor("memcell_with_nanoantennav2_copy.net")

In [5]:
raw, log = runner.run_now(netlist, run_filename="Test_sim2")

In [6]:
raw = RawRead("SimResults/Test_sim2.raw")

In [6]:
print(raw.get_trace_names())

['time', 'V(n003)', 'V(n004)', 'V(n005)', 'V(n001)', 'V(n002)', 'V(readref)', 'V(p001)', 'I(Store1)', 'I(Read1)', 'I(Set1)', 'I(C7)', 'I(R5)', 'I(Read)', 'Ia(T1)', 'Ib(T1)', 'Ic(T1)', 'Id(T1)', 'Ix(x1:VIN)', 'Ix(x1:E)', 'Ix(x1:C)', 'Ix(x2:VIN)', 'Ix(x2:E)', 'Ix(x2:C)']


In [39]:
print(raw.get_axis())

[  0.          3.0041943   6.0761943   9.1481943  12.2201943  15.2921943
  18.3641943  21.4361943  24.5081943  27.5801943  30.6521943  33.7241943
  36.7961943  39.8681943  42.9401943  46.0121943  49.0841943  52.1561943
  55.2281943  58.3001943  61.3721943  64.4441943  67.5161943  70.5881943
  73.6601943  76.7321943  79.8041943  82.8761943  85.9481943  89.0201943
  92.0921943  95.1641943  98.2361943 101.3081943 104.3801943 107.4521943
 110.5241943 113.5961943 116.6681943 119.7401943 122.8121943 125.8841943
 128.9561943 132.0281943 135.1001943 138.1721943 141.2441943 144.3161943
 147.3881943 150.4601943 153.5321943 156.6041943 159.6761943 162.7481943
 165.8201943 168.8921943 171.9641943 175.0361943 178.1081943 181.1801943
 184.2521943 187.3241943 190.3961943 193.4681943 196.5401943 199.6121943
 202.6841943 205.7561943 208.8281943 211.9001943 214.9721943 218.0441943
 221.1161943 224.1881943 227.2601943 230.3321943 233.4041943 236.4761943
 239.5481943 242.6201943 245.6921943 248.7641943 25

In [7]:
def picking_time_frames(time, write_0, write_1):
    """ Looks at values in write_0 and write_1 and return the time intervals in which the values are non zero.
        Returns a dictionary with keys write_0 and write_1 with corresponding time intervals as tuples."""
    def find_intervals(signal):
        intervals = []
        start_idx = None
        for i, value in enumerate(signal):
            if value != 0 and start_idx is None:
                start_idx = i
            elif value == 0 and start_idx is not None:
                intervals.append((time[start_idx], time[i-1]))
                start_idx = None
        if start_idx is not None:
            intervals.append((time[start_idx], time[-1]))
        return intervals
    return {
        'write_0': find_intervals(write_0),
        'write_1': find_intervals(write_1)
    }
def calc_charges(time_intervals, time, current):
    """takes a dictionary with keys of corresponding time intervals and integrates current in those intervals.
        Returns another dictionary with write_1 and write_0 keys and list of corresponding charges."""
    charges = {'write_0': [], 'write_1': []}
    for key in ['write_0', 'write_1']:
        for start_time, end_time in time_intervals.get(key, []):
            #identify indices corresponding to the current interval
            indices = np.where((time >= start_time) & (time <= end_time))[0]
            if len(indices) > 1:
                #extract the time and current values for the interval
                interval_time = time[indices]
                interval_current = current[indices]
                #integrate
                charge = simpson(interval_current, x=interval_time)
                charges[key].append(charge)
            else:
                #if not enough points are found, append zero charge
                charges[key].append(0.0)
    return charges
def run_sim(output_folder, netlist, output_filename, params):
    """runs simulation for one iteration and returns a dictionary with time, write_1 voltage, write_0 voltage, and currents labeled respectively.
    output_folder: name of folder that you want to ouput into, str
    netlist: name of netlist, str
    output_filename: name of file that it outputs that you want, str
    params: parameters that are changed, list of list, [name of param in str, number to change it to in float]
    """
    out_path = "./" + output_folder
    runner = SimRunner(output_folder=out_path, simulator=LTspice)
    net = SpiceEditor(netlist)
    # If we have parameters, collect them into a dictionary.
    if len(params) != 0:
        param_dict = {p[0]: p[1] for p in params}
        net.set_parameters(**param_dict)
        net.save_netlist("new" + netlist)
    # Run the simulation
    raw, log = runner.run_now(netlist, run_filename=output_filename)
    full_path_after_run = output_folder + "/" + output_filename + ".raw"
    raw = RawRead(full_path_after_run)
    readref = raw.get_trace("V(readref)").get_wave()
    n005 = raw.get_trace("V(n005)").get_wave()
    V_write_0 = -readref + n005
    return {
        "time": raw.get_trace("time").get_wave(),
        "Write_1": raw.get_trace("V(n001)").get_wave(),
        "Write_0": V_write_0,
        "currents": raw.get_trace("I(Read)").get_wave()
    }
def objective_func(sim_res):
    """sim_res: dictionary returned from run_sim. Returns charge difference between read 1 and 0."""
    time = sim_res["time"]
    write_1 = sim_res["Write_1"]
    write_0 = sim_res["Write_0"]
    currents = sim_res["currents"]
    time_ints = picking_time_frames(time, write_0, write_1)
    charges = calc_charges(time_ints, time, currents)
    total_write_0 = charges['write_0'][0]
    total_write_1 = charges['write_1'][0]
    return total_write_1 - total_write_0

In [60]:
#run_sim test
print(run_sim("SimResults", "memcell_with_nanoantennav2_copy.net", "Test_sim1", [["amp", 15e-12]]))

{'time': array([  0.       ,   3.0041943,   6.0761943,   9.1481943,  12.2201943,
        15.2921943,  18.3641943,  21.4361943,  24.5081943,  27.5801943,
        30.6521943,  33.7241943,  36.7961943,  39.8681943,  42.9401943,
        46.0121943,  49.0841943,  52.1561943,  55.2281943,  58.3001943,
        61.3721943,  64.4441943,  67.5161943,  70.5881943,  73.6601943,
        76.7321943,  79.8041943,  82.8761943,  85.9481943,  89.0201943,
        92.0921943,  95.1641943,  98.2361943, 101.3081943, 104.3801943,
       107.4521943, 110.5241943, 113.5961943, 116.6681943, 119.7401943,
       122.8121943, 125.8841943, 128.9561943, 132.0281943, 135.1001943,
       138.1721943, 141.2441943, 144.3161943, 147.3881943, 150.4601943,
       153.5321943, 156.6041943, 159.6761943, 162.7481943, 165.8201943,
       168.8921943, 171.9641943, 175.0361943, 178.1081943, 181.1801943,
       184.2521943, 187.3241943, 190.3961943, 193.4681943, 196.5401943,
       199.6121943, 202.6841943, 205.7561943, 208.82819

In [10]:
#gradient descent
def grad_desc(netlist, output_folder, output_filename, iterations, params, delta, learn_rate, margin):
    count = 0
    for iteration in range(iterations):
        #run original netlist to find objective
        diffs = []
        new_param_vals = []
        original_params = copy.deepcopy(params)
        orig_sim_res = run_sim(output_folder, netlist, output_filename, params)
        f_current_value = objective_func(orig_sim_res)
        for seq in range(len(params)):
            #f_plus
            params[seq][1] = original_params[seq][1] + delta
            plus_sim_res = run_sim(output_folder, netlist, output_filename, params)
            f_plus = objective_func(plus_sim_res)
            #f_minus
            params[seq][1] = original_params[seq][1] - delta
            minus_sim_res = run_sim(output_folder, netlist, output_filename, params)
            f_minus = objective_func(minus_sim_res)
            #calculating grads and updating params
            params[seq][1] = original_params[seq][1]
            grad = (f_plus - f_minus)/(2.0 * delta)
            added = learn_rate*grad
            diffs.append(added)
            new_param_vals.append(original_params[seq][1] + (added))
            count+=1
            print("iteration num:" + str(count) + " " + str(params[seq][0]) + ":" + str(original_params[seq][1] + (added)))
        for ind in range(len(params)):
            params[ind][1] = new_param_vals[ind]
        #convergence factor
        for diff in diffs:
            if abs(diff) < margin:
                return params
    return params
        
            
            
        
            
    

In [12]:
print(grad_desc("memcell_with_nanoantennav2_copy.net", 'SimResults', "Test_sim1", 10, [["amp", 15e-12],["sigdur", 5.6]], 1e-12, 1, 1e-15))

iteration num:1 amp:1.5e-11
iteration num:2 sigdur:5.6
[['amp', np.float64(1.5e-11)], ['sigdur', np.float64(5.6)]]
